[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/QuantLet/VaR_CEE_FM/blob/main/VaR_CEE_Chronos/VaR_CEE_Chronos.ipynb)

# VaR_CEE_Chronos

Generates zero-shot VaR and ES forecasts using the Chronos-2 foundation model (Amazon).
Produces 1000 Monte Carlo forecast samples per date using a 512-day context window
and derives VaR/ES from the empirical distribution of samples.

**Published in:** *Can Foundation Models Manage Risk? Zero-Shot VaR and ES Forecasting for CEE Markets*

**Author:** Daniel Traian Pele

**R1 update:** 1000 samples (was 200), context 512 (was 250), stride 1 (full daily run).

In [ ]:
# Install dependencies
!pip install -q chronos-forecasting torch pandas numpy matplotlib

In [ ]:
import time
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from pathlib import Path
from google.colab import files

np.random.seed(42)
torch.manual_seed(42)

# Check GPU
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")
if device == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
# ── Configuration ──────────────────────────────────────────
MARKETS = {
    "Romania":  {"index": "BET",   "fx": "EURRON"},
    "Poland":   {"index": "WIG20", "fx": "EURPLN"},
    "Czechia":  {"index": "PX",    "fx": "EURCZK"},
    "Hungary":  {"index": "BUX",   "fx": "EURHUF"},
    "Bulgaria": {"index": "SOFIX", "fx": "USDBGN"},
}

OOS_START = "2018-01-01"
VAR_ALPHAS = [0.01, 0.025, 0.05]
ES_ALPHA = 0.025
FM_CONTEXT = 512       # R1: was 250
FM_NUM_SAMPLES = 1000  # R1: was 200
FM_HORIZON = 1
STRIDE = 1             # R1: was 5 — full daily run

# Output directory
OUT_DIR = Path("results")
OUT_DIR.mkdir(exist_ok=True)

def get_all_series():
    series = []
    for country, info in MARKETS.items():
        series.append((country, info["index"], f"{info['index']}_ret", "index"))
        series.append((country, info["fx"], f"{info['fx']}_ret", "fx"))
    return series

print("Series to process:")
for c, raw_id, sid, st in get_all_series():
    print(f"  {c}: {sid} ({st})")

## 1. Upload Raw Data

Upload the 10 CSV files from `data/raw/`:
- `BET.csv`, `WIG20.csv`, `PX.csv`, `BUX.csv`, `SOFIX.csv`
- `EURRON.csv`, `EURPLN.csv`, `EURCZK.csv`, `EURHUF.csv`, `USDBGN.csv`

Each CSV must have columns: `Date, Open, High, Low, Close, Volume`

In [ ]:
# Upload CSV files
print("Please upload the 10 raw CSV files (BET.csv, WIG20.csv, PX.csv, BUX.csv, SOFIX.csv,")
print("EURRON.csv, EURPLN.csv, EURCZK.csv, EURHUF.csv, USDBGN.csv):")
uploaded = files.upload()
print(f"\nUploaded {len(uploaded)} files: {list(uploaded.keys())}")

In [ ]:
# ── Compute log returns from raw CSVs ─────────────────────
returns_dict = {}

for country, raw_id, series_id, stype in get_all_series():
    csv_file = f"{raw_id}.csv"
    if csv_file not in uploaded:
        print(f"  WARNING: {csv_file} not uploaded, skipping {series_id}")
        continue

    df = pd.read_csv(csv_file, parse_dates=["Date"], index_col="Date")
    df = df.sort_index()
    # Log returns from Close prices
    log_ret = np.log(df["Close"] / df["Close"].shift(1)).dropna()
    log_ret.name = series_id
    returns_dict[series_id] = log_ret
    print(f"  {series_id}: {len(log_ret)} observations, "
          f"{log_ret.index[0].date()} to {log_ret.index[-1].date()}")

print(f"\nTotal series loaded: {len(returns_dict)}")

## 2. Load Chronos-2 Model

In [ ]:
from chronos import ChronosPipeline

print("Loading Chronos-2 model...")
try:
    pipeline = ChronosPipeline.from_pretrained(
        "amazon/chronos-t5-base",  # 200M params
        device_map=device,
        torch_dtype=torch.float32,
    )
    print(f"Chronos-2 (base, 200M) loaded on {device}")
except Exception as e:
    print(f"Base model failed: {e}")
    print("Trying small model as fallback...")
    pipeline = ChronosPipeline.from_pretrained(
        "amazon/chronos-t5-small",
        device_map=device,
        torch_dtype=torch.float32,
    )
    print(f"Chronos-2 (small) loaded on {device}")

## 3. Run VaR/ES Forecasting

In [ ]:
def forecast_chronos(pipeline, context, num_samples=1000, chunk_size=200):
    """Generate forecast samples from Chronos-2, batched to avoid OOM."""
    context_tensor = torch.tensor(context, dtype=torch.float32).unsqueeze(0)
    all_samples = []
    remaining = num_samples
    while remaining > 0:
        n = min(remaining, chunk_size)
        with torch.no_grad():
            forecast = pipeline.predict(
                context_tensor,
                prediction_length=FM_HORIZON,
                num_samples=n,
            )
        all_samples.append(forecast[0, :, 0].cpu().numpy())
        remaining -= n
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
    return np.concatenate(all_samples)  # (num_samples,)


def extract_var_es(samples):
    """Extract VaR at 1%, 2.5%, 5% and ES at 2.5% from samples.
    With 1000 samples: 5% = 50th smallest, 2.5% = 25th, 1% = 10th.
    """
    row = {}
    for alpha in VAR_ALPHAS:
        alpha_str = {0.01: "01", 0.025: "025", 0.05: "05"}[alpha]
        row[f"var_{alpha_str}"] = float(np.quantile(samples, alpha))

    var_025 = row["var_025"]
    tail = samples[samples <= var_025]
    if len(tail) >= 3:
        row["es_025"] = float(np.mean(tail))
    else:
        k = max(5, int(0.025 * len(samples)))
        row["es_025"] = float(np.mean(np.sort(samples)[:k]))

    row["fm_median"] = float(np.median(samples))
    row["fm_std"] = float(np.std(samples))
    return row


def run_chronos(pipeline, returns, oos_dates, series_id, save_path):
    """Run Chronos-2 VaR/ES forecasts for one series with incremental saving."""
    results = []
    n_total = len(oos_dates)

    for i, date in enumerate(oos_dates):
        loc = returns.index.get_loc(date)
        if loc < FM_CONTEXT:
            continue

        context = returns.iloc[loc - FM_CONTEXT:loc].values.astype(np.float64)
        samples = forecast_chronos(pipeline, context, FM_NUM_SAMPLES)

        y_true = float(returns.iloc[loc])
        row = {"date": str(date.date()), "y_true": y_true}
        row.update(extract_var_es(samples))
        results.append(row)

        # incremental save every 100 forecasts
        if len(results) % 100 == 0:
            pd.DataFrame(results).to_csv(save_path, index=False, float_format="%.8f")

        if (i + 1) % 100 == 0 or i == n_total - 1:
            print(f"    [{i+1}/{n_total}] {series_id}")

    df = pd.DataFrame(results)
    df.to_csv(save_path, index=False, float_format="%.8f")
    return df

In [ ]:
# ── Main forecasting loop ─────────────────────────────────
print("=" * 60)
print("CHRONOS-2 VaR/ES FORECASTING")
print(f"Context: {FM_CONTEXT} | Samples: {FM_NUM_SAMPLES} | Stride: {STRIDE}")
print("=" * 60)

all_results = {}

for country, raw_id, series_id, stype in get_all_series():
    if series_id not in returns_dict:
        print(f"\nSKIP: {series_id} (not loaded)")
        continue

    returns = returns_dict[series_id]
    oos_mask = returns.index >= pd.Timestamp(OOS_START)
    oos_dates = returns.index[oos_mask][::STRIDE]

    out_path = OUT_DIR / f"Chronos-2_{series_id}.csv"

    # Resume support: skip if already complete
    if out_path.exists():
        existing = pd.read_csv(out_path)
        if len(existing) >= len(oos_dates):
            print(f"\n[{series_id}] Already done ({len(existing)} rows), skipping.")
            all_results[series_id] = existing
            continue

    print(f"\n[{series_id}] {len(returns)} total obs, {len(oos_dates)} OOS dates")

    t0 = time.time()
    df_result = run_chronos(pipeline, returns, oos_dates, series_id, out_path)
    elapsed = time.time() - t0

    all_results[series_id] = df_result
    print(f"  Saved {out_path.name} ({len(df_result)} rows) in {elapsed:.1f}s")

print("\n" + "=" * 60)
print("DONE")
print("=" * 60)

## 4. Visualisation — VaR Breach Plot

In [ ]:
# Plot VaR breach examples for all index series
for country, raw_id, series_id, stype in get_all_series():
    if stype != "index" or series_id not in all_results:
        continue

    df_plot = all_results[series_id].copy()
    if "date" in df_plot.columns:
        df_plot["date"] = pd.to_datetime(df_plot["date"])
    var_col = "var_01"
    if var_col not in df_plot.columns:
        continue

    fig, ax = plt.subplots(figsize=(14, 4))
    ax.plot(df_plot["date"], df_plot["y_true"], linewidth=0.4,
            color="black", label="Returns")
    ax.plot(df_plot["date"], df_plot[var_col], linewidth=0.6,
            color="coral", linestyle="--", label="Chronos-2 VaR 1%")

    breaches = df_plot["y_true"] < df_plot[var_col]
    if breaches.any():
        ax.scatter(df_plot.loc[breaches, "date"],
                   df_plot.loc[breaches, "y_true"],
                   color="red", s=10, zorder=5, alpha=0.7, label="Breach")

    n_breach = breaches.sum()
    pct_breach = 100 * n_breach / len(df_plot)
    ax.set_title(f"Chronos-2 VaR 1% — {series_id}  "
                 f"(breaches: {n_breach}/{len(df_plot)} = {pct_breach:.1f}%)")
    ax.set_ylabel("Log returns")
    ax.legend(loc="lower center", bbox_to_anchor=(0.5, -0.25),
              ncol=3, frameon=False)
    ax.axhline(y=0, color="gray", linewidth=0.3)
    plt.tight_layout()
    plt.show()

## 5. Download Results

In [ ]:
# Download all result CSVs
import zipfile, io

zip_buffer = io.BytesIO()
with zipfile.ZipFile(zip_buffer, "w", zipfile.ZIP_DEFLATED) as zf:
    for f in sorted(OUT_DIR.glob("Chronos-2_*.csv")):
        zf.write(f, f.name)

zip_buffer.seek(0)
with open("Chronos2_results.zip", "wb") as fout:
    fout.write(zip_buffer.read())

files.download("Chronos2_results.zip")
print("Download started: Chronos2_results.zip")